In [1]:
from datasets import load_dataset
dataset = load_dataset("openbmb/RLAIF-V-Dataset", split="train[:1%]")
sample = dataset[1]
sample["image"].show()
sample["question"]
sample["rejected"]
sample["chosen"]


/home/xuejunzh/anaconda3/envs/solo/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'The image shows a Union Organization table setup with 18,000 families.'

Error: no "view" rule for type "image/png" passed its test case
       (for more information, add "--debug=1" on the command line)


In [2]:
sample = dataset[1]

# Display the image
sample["image"].show()

Error: no "view" rule for type "image/png" passed its test case
       (for more information, add "--debug=1" on the command line)


In [3]:
from datasets import features
from transformers import AutoProcessor

In [4]:
processor = AutoProcessor.from_pretrained("HuggingFaceM4/idefics2-8b", do_image_splitting=False)

Chat templates should be in a 'chat_template.json' file but found key='chat_template' in the processor's config. Make sure to move your template to its own file.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [5]:
def format(example):
    # Prepare the input for the chat template
    prompt = [
        {
            "role": "user",
            "content": [{"type": "image"}, {"type": "text", "text": example["question"]}],
        },
    ]
    chosen = [
        {
            "role": "assistant",
            "content": [{"type": "text", "text": example["chosen"]}],
        },
    ]
    rejected = [
        {
            "role": "assistant",
            "content": [{"type": "text", "text": example["rejected"]}],
        },
    ]
    
    prompt = processor.apply_chat_template(prompt, tokenize=False)
    chosen = processor.apply_chat_template(chosen, tokenize=False)
    rejected = processor.apply_chat_template(rejected, tokenize=False)
    # Resize the image to ensure it fits within the maximum allowable
    # size of the processor to prevent OOM errors.
    max_size = processor.image_processor.size["longest_edge"]
    example["image"].thumbnail((max_size, max_size))
    return {"images": [example["image"]], "prompt": prompt, "chosen": chosen, "rejected": rejected}

In [6]:
dataset = dataset.map(format, remove_columns=dataset.column_names)

# Make sure that the images are decoded, it prevents from storing bytes.
# More info here https://github.com/huggingface/blog/pull/2148#discussion_r1667400478
f = dataset.features
f["images"] = features.Sequence(features.Image(decode=True))  # to avoid bytes
dataset = dataset.cast(f)

In [7]:
dataset[1]

{'chosen': 'Assistant: The image shows a Union Organization table setup with 18,000 families.<end_of_utterance>\n',
 'rejected': 'Assistant: The image does not provide any information about families.<end_of_utterance>\n',
 'images': [<PIL.JpegImagePlugin.JpegImageFile image mode=L size=980x812>],
 'prompt': 'User:<image>how many families?<end_of_utterance>\n'}

In [8]:
import torch
from transformers import AutoModelForVision2Seq

model = AutoModelForVision2Seq.from_pretrained("HuggingFaceM4/idefics2-8b", torch_dtype=torch.bfloat16)


Loading checkpoint shards: 100%|██████████| 7/7 [00:02<00:00,  2.84it/s]


In [9]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",  
    num_train_epochs=1,    
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=16,   
    warmup_steps=500,             
    weight_decay=0.01,           
    logging_dir="./logs",            
    logging_steps=10,
    bf16=True                       
)

In [10]:
from transformers import AutoModelForVision2Seq

model = AutoModelForVision2Seq.from_pretrained("HuggingFaceM4/idefics2-8b")


Loading checkpoint shards: 100%|██████████| 7/7 [00:00<00:00,  9.99it/s]


In [11]:
from transformers import AutoModelForVision2Seq
from peft import get_peft_model, LoraConfig

In [12]:
model = AutoModelForVision2Seq.from_pretrained("HuggingFaceM4/idefics2-8b")

Loading checkpoint shards: 100%|██████████| 7/7 [00:00<00:00, 10.15it/s]


In [13]:
peft_config = LoraConfig(target_modules="all-linear")

In [14]:
model = get_peft_model(model, peft_config)

In [15]:
model.print_trainable_parameters()

trainable params: 27,674,368 || all params: 8,430,442,480 || trainable%: 0.3283


In [16]:
from datasets import features, load_dataset
from transformers import AutoModelForVision2Seq, AutoProcessor
import torch
from trl import DPOConfig, DPOTrainer
from peft import LoraConfig


In [17]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [38]:
model = AutoModelForVision2Seq.from_pretrained("HuggingFaceM4/idefics2-8b", torch_dtype=torch.bfloat16)
processor = AutoProcessor.from_pretrained("HuggingFaceM4/idefics2-8b", do_image_splitting=False)

# Load the dataset
dataset = load_dataset("openbmb/RLAIF-V-Dataset", split="train")

Loading checkpoint shards: 100%|██████████| 7/7 [00:07<00:00,  1.05s/it]
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [40]:
def format(example):
    # Prepare the input for the chat template
    prompt = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": example["question"]}]}]
    chosen = [{"role": "assistant", "content": [{"type": "text", "text": example["chosen"]}]}]
    rejected = [{"role": "assistant", "content": [{"type": "text", "text": example["rejected"]}]}]
    # Apply the chat template
    prompt = processor.apply_chat_template(prompt, tokenize=False)
    chosen = processor.apply_chat_template(chosen, tokenize=False)
    rejected = processor.apply_chat_template(rejected, tokenize=False)
    # Resize the image to ensure it fits within the maximum allowable
    # size of the processor to prevent OOM errors.
    max_size = processor.image_processor.size["longest_edge"] // 2
    example["image"].thumbnail((max_size, max_size))
    return {"images": [example["image"]], "prompt": prompt, "chosen": chosen, "rejected": rejected}

    # Apply the formatting function to the dataset
dataset = dataset.map(format, remove_columns=dataset.column_names, num_proc=32)  

Map (num_proc=32): 100%|██████████| 83132/83132 [02:06<00:00, 659.51 examples/s] 


In [41]:
f = dataset.features
f["images"] = features.Sequence(features.Image(decode=True))
dataset = dataset.cast(f)

Casting the dataset: 100%|██████████| 83132/83132 [00:03<00:00, 27515.36 examples/s]


In [44]:
training_args = DPOConfig(
        output_dir="idefics2-8b-dpo",
        bf16=True,
        gradient_checkpointing=True,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=32,
        num_train_epochs=1,
        dataset_num_proc=8,  
        dataloader_num_workers=8, 
        logging_steps=10,
    )
trainer = DPOTrainer(
        model,
        ref_model=None,  # not needed when using peft
        args=training_args,
        train_dataset=dataset,
        tokenizer=processor,
        peft_config=LoraConfig(target_modules="all-linear"),
    )

trainer.train()

/home/xuejunzh/anaconda3/envs/solo/lib/python3.10/site-packages/trl/trainer/dpo_trainer.py:671: UserWarning: `max_length` is not set in the DPOConfig's init it will default to `512` by default, but you should do it yourself in the future.
  warnings.warn(
/home/xuejunzh/anaconda3/envs/solo/lib/python3.10/site-packages/trl/trainer/dpo_trainer.py:684: UserWarning: `max_prompt_length` is not set in the DPOConfig's init it will default to `128` by default, but you should do it yourself in the future.
  warnings.warn(
/home/xuejunzh/anaconda3/envs/solo/lib/python3.10/site-packages/trl/trainer/dpo_trainer.py:719: UserWarning: When using DPODataCollatorWithPadding, you should set `remove_unused_columns=False` in your TrainingArguments we have set it for you, but you should do it yourself in the future.
  warnings.warn(
Tokenizing train dataset (num_proc=8): 100%|██████████| 83132/83132 [09:17<00:00, 149.16 examples/s]
wandb: WARNING The `run_name` is currently set to the same value as `Traini

/home/xuejunzh/anaconda3/envs/solo/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
/home/xuejunzh/anaconda3/envs/solo/lib/python3.10/site-packages/torch/_dynamo/eval_frame.py:600: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.

OutOfMemoryError: CUDA out of memory. Tried to allocate 32.00 MiB. GPU 0 has a total capacity of 44.55 GiB of which 13.81 MiB is free. Process 1907832 has 13.86 GiB memory in use. Including non-PyTorch memory, this process has 30.04 GiB memory in use. Process 2528893 has 522.00 MiB memory in use. Of the allocated memory 29.38 GiB is allocated by PyTorch, and 180.70 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)